# Chapitre 11 - Cours 3 : nombre d'unités, weight decay et chiffres manuscrits

Ce notebook reproduit les deux exemples des sections 11.6 et 11.7 du livre :

1. les **données simulées** « somme de sigmoïdes » et « radiale » : erreur de test en fonction du
   nombre d'unités cachées et du paramètre de weight decay (figures 11.6, 11.7 et 11.8) ;
2. les **chiffres manuscrits** (codes postaux) : architectures du tableau 11.1, ajustement des
   réseaux entièrement connectés Net-1 et Net-2 sur un petit échantillon et courbes de
   performance au fil des époques (figure 11.11).

Il se termine par la discussion de la section 11.8 (forces et limites des réseaux).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch11_reseaux import MLP, n_weights, one_hot, sigmoid

## 1. Données simulées (section 11.6)

On simule deux modèles additifs $Y = f(X) + \varepsilon$ :

$$
\begin{aligned}
\text{somme de sigmoïdes : }\quad Y &= \sigma(a_1^T X) + \sigma(a_2^T X) + \varepsilon_1, \qquad p = 2,\ a_1 = (3,3)^T,\ a_2 = (3,-3)^T ;\\
\text{radial : }\quad Y &= \prod_{m=1}^{10} \phi(X_m) + \varepsilon_2, \qquad p = 10,\ \phi(t) = (2\pi)^{-1/2} e^{-t^2/2}.
\end{aligned}
$$

Les $X_j$ sont des gaussiennes centrées réduites indépendantes et les bruits sont gaussiens, de
variance choisie pour que le **rapport signal sur bruit**

$$\frac{\mathrm{Var}\big(\mathbb E(Y\mid X)\big)}{\mathrm{Var}\big(Y - \mathbb E(Y \mid X)\big)} = \frac{\mathrm{Var}(f(X))}{\mathrm{Var}(\varepsilon)} \qquad \text{(éq. (11.18))}$$

vaille 4 dans les deux cas. On tire $N = 100$ observations d'apprentissage et $10\,000$ de test.

Pour le modèle radial, la variance de $f$ se calcule exactement. Par indépendance des $X_m$,
$\mathbb E f(X) = \prod_m \mathbb E\phi(X_m) = \big(\mathbb E\phi(X_1)\big)^{10}$ et
$\mathbb E f(X)^2 = \big(\mathbb E\phi(X_1)^2\big)^{10}$. Or, pour $X_1 \sim \mathcal N(0,1)$,

$$\mathbb E\,\phi(X_1) = \int \phi(t)^2\,dt = \frac{1}{2\pi}\int e^{-t^2} dt = \frac{\sqrt\pi}{2\pi} = \frac{1}{2\sqrt\pi},
\qquad \mathbb E\,\phi(X_1)^2 = \int \phi(t)^3 dt = \frac{1}{(2\pi)^{3/2}} \sqrt{\frac{2\pi}{3}} = \frac{1}{2\pi\sqrt 3},$$

(en utilisant $\int e^{-ct^2}dt = \sqrt{\pi/c}$), d'où
$\mathrm{Var} f(X) = (2\pi\sqrt3)^{-10} - (2\sqrt\pi)^{-20}$.